# Guia completo da `dashgusbr`

**Tudo que a biblioteca faz, em um notebook**: análise e visualização do histórico
completo do Campeonato Brasileiro (1971–hoje), sobre a OBT publicada pelo pipeline
[Infra-Brasileirao](https://github.com/gustavogasperetti/Infra-Brasileirao).

Roteiro:

1. [Carregando os dados](#1)
2. [Explorando a base](#2)
3. [Classificação](#3)
4. [Evolução de pontos e corrida pelo título](#4)
5. [Histórico de um clube](#5)
6. [Confrontos diretos](#6)
7. [Retrospecto contra todos os adversários](#7)
8. [Casa × fora, sequências, forma e resumo](#8)
9. [O campeonato em números (visão macro)](#9)
10. [Geografia: estados e mapa do Brasil](#10)
11. [Rankings e líderes](#11)
12. [Cores oficiais dos clubes](#12)
13. [Personalização: todas as combinações](#13)
14. [Exportação (HTML e imagem)](#14)
15. [Qualidade dos dados](#15)
16. [Uso avançado: as camadas puras](#16)
17. [CLI: a lib no terminal](#17)

> Instalação: `pip install dashgusbr` (Python ≥ 3.9; depende só de `pandas` e `plotly`).
> Para exportar imagens estáticas: `pip install dashgusbr[imagem]`.

<a id="1"></a>
## 1. Carregando os dados

A classe `Brasileirao` é a porta de entrada. Ela baixa a OBT na **primeira consulta**
(não no construtor), guarda em memória e em disco (`~/.dashgusbr/cache`, validade 24h)
— as próximas sessões carregam do cache e funcionam offline.

In [ ]:
from dashgusbr import Brasileirao

br = Brasileirao()          # fonte "auto": CSV no GitHub -> fallback Google Sheets
br                          # repr mostra o estado (nada foi baixado ainda)

Outras formas de construir, para reprodutibilidade ou trabalho offline:

```python
br = Brasileirao(fonte="github")           # força o CSV do GitHub
br = Brasileirao(fonte="sheets")           # força o Google Sheets
br = Brasileirao(fonte="dados/obt.csv")    # CSV local no mesmo schema (snapshot congelado)
br = Brasileirao(github_url="https://raw.githubusercontent.com/.../obt.csv")
br.recarregar()                            # força novo download, ignorando caches
```

Para acompanhar o download/cache/fallback, ligue o logging:

In [ ]:
import logging

logging.basicConfig(level=logging.INFO)    # logger "dashgusbr" reporta o progresso
print(f"{len(br.partidas())} partidas carregadas")   # primeira consulta dispara a carga
logging.disable(logging.INFO)              # silencia daqui em diante (opcional)

<a id="2"></a>
## 2. Explorando a base

Métodos utilitários para saber **o que existe** na base. Nomes de time aceitam
variações de caixa, acento e hífen em toda a biblioteca ("gremio" → "Grêmio");
um nome desconhecido vira erro com sugestões — nunca um palpite silencioso.

In [ ]:
anos = br.anos()
print(f"{len(anos)} temporadas: {anos[0]}–{anos[-1]}")
print(f"{len(br.times())} clubes na história; em 2023 foram {len(br.times(ano=2023))}")

In [ ]:
br.partidas(ano=2023, time="gremio").head(3)   # nome tolerante: resolve para "Grêmio"

In [ ]:
# Nome errado -> erro com sugestões (descomente para ver):
# br.partidas(time="Fluminse")
# ValueError: Time 'Fluminse' não encontrado na base. Você quis dizer: Fluminense, ...?

br.df.head(3)    # a OBT completa, no schema canônico (não modifique in-place)

<a id="3"></a>
## 3. Classificação

`tabela(ano)` devolve o DataFrame; `plot_tabela(ano)` o gráfico. Só jogos de
**pontos corridos** contam. Os pontos vêm da regra da época (vitória = 2 pts até
1994, 3 pts depois) e o **aproveitamento é normalizado** — a única métrica
comparável entre eras.

In [ ]:
br.tabela(2023).head(6)

In [ ]:
br.plot_tabela(2023)

In [ ]:
# A mesma tabela de uma era antiga: pontos na regra de 2 pts por vitória
br.tabela(1971).head(4)

<a id="4"></a>
## 4. Evolução de pontos e corrida pelo título

- `plot_evolucao(times, ano)` — um ou vários times (máx. 8 séries);
- `plot_corrida_titulo(ano, n=4)` — atalho: os `n` primeiros da tabela final;
- `evolucao(time, ano)` — o DataFrame jogo a jogo por trás do gráfico.

In [ ]:
br.plot_evolucao("Botafogo", 2023)             # um time só: sem legenda, rótulo direto

In [ ]:
br.plot_evolucao(["Palmeiras", "Botafogo", "Grêmio"], 2023)   # várias séries

In [ ]:
br.plot_corrida_titulo(2023, n=4, cores_times=True)   # G4 com as cores oficiais

In [ ]:
br.evolucao("Palmeiras", 2023).tail(3)   # o DataFrame por trás: jogo, adversário, acumulado

<a id="5"></a>
## 5. Histórico de um clube

`historico(time)` = campanha por temporada (posição, pontos, aproveitamento).
`plot_historico` plota qualquer métrica dessas colunas.

In [ ]:
br.historico("Santos").tail(5)

In [ ]:
br.plot_historico("Santos")                          # aproveitamento (padrão)

In [ ]:
br.plot_historico("Santos", metrica="posicao", cores_times=True)   # outra métrica

<a id="6"></a>
## 6. Confrontos diretos

- `confronto(a, b)` — dict-resumo + DataFrame `partidas` com todos os jogos;
- `plot_confronto(a, b)` — vitórias/empates lado a lado;
- `plot_confronto_evolucao(a, b)` — **quem abriu vantagem na história**: saldo
  acumulado do confronto ao longo dos anos (positivo = vantagem do primeiro time).

In [ ]:
resumo = br.confronto("Flamengo", "Palmeiras")
{k: v for k, v in resumo.items() if k != "partidas"}

In [ ]:
br.plot_confronto("Flamengo", "Palmeiras", cores_times=True)

In [ ]:
br.plot_confronto_evolucao("Grêmio", "Internacional", cores_times=True)   # o GreNal na linha do tempo

In [ ]:
br.confronto_evolucao("Grêmio", "Internacional").tail(3)   # o DataFrame por trás

<a id="7"></a>
## 7. Retrospecto contra todos os adversários

`contra(time)` agrega o retrospecto contra **cada** rival (todas as fases);
`plot_contra` mostra o aproveitamento contra os mais enfrentados.

In [ ]:
br.contra("Palmeiras", min_jogos=30).head(8)   # min_jogos descarta confrontos raros

In [ ]:
br.plot_contra("Palmeiras", top=12)

<a id="8"></a>
## 8. Casa × fora, sequências, forma e resumo

Quatro leituras do desempenho de um clube:

In [ ]:
br.casa_fora("Grêmio")            # V/E/D e aproveitamento como mandante x visitante

In [ ]:
br.plot_casa_fora("Grêmio")

In [ ]:
br.casa_fora("Grêmio", ano=2023)  # o mesmo recorte, restrito a uma temporada

In [ ]:
br.sequencias("Flamengo")         # maiores sequências da história (com período)

In [ ]:
forma = br.forma("Botafogo", n=5)  # os últimos 5 jogos
print(f"Aproveitamento no período: {forma.attrs['aproveitamento']}%")
forma

In [ ]:
br.resumo("Cruzeiro")             # cartão-resumo: campanhas, recordes, totais

<a id="9"></a>
## 9. O campeonato em números (visão macro)

Análises agregadas da competição inteira — 50+ anos de futebol em um gráfico.

In [ ]:
br.plot_gols_por_temporada()      # a média de gols caiu ao longo das décadas?

In [ ]:
br.gols_por_decada()              # a mesma pergunta por década + era de pontuação

In [ ]:
br.plot_mandante_visitante()      # o fator casa está encolhendo?

In [ ]:
br.plot_placares()                # heatmap: 1x0 e 1x1 dominam a história

In [ ]:
br.plot_placares(ano=2023, max_gols=4)   # recorte de uma temporada, matriz menor

In [ ]:
br.plot_saldos()                  # a assimetria da distribuição É o fator casa

In [ ]:
br.goleadas(10)                   # as 10 maiores goleadas da história

In [ ]:
br.classicos()                    # clássico estadual muda o jogo? (gols, empates, fator casa)

In [ ]:
br.fases()                        # mata-mata x pontos corridos

In [ ]:
br.viagem()                       # o visitante sofre mais quando cruza o estado?

<a id="10"></a>
## 10. Geografia: estados e mapa do Brasil

Usa `estado_mandante`/`estado_visitante`. O mapa coroplético baixa um GeoJSON
público das UFs na primeira vez (e cacheia em disco por 30 dias).

In [ ]:
br.estados().head(8)              # jogos, gols, clubes e fator casa por UF

In [ ]:
br.plot_estados()

In [ ]:
br.plot_mapa_estados()            # metrica="jogos" (padrão)

In [ ]:
br.plot_mapa_estados(metrica="media_gols", titulo="Onde saem mais gols?")

<a id="11"></a>
## 11. Rankings e líderes

> **Ressalva histórica**: até 2002 o líder dos pontos corridos **não** é
> necessariamente o campeão (a base não decide o mata-mata final). A partir de
> 2003, líder = campeão.

In [ ]:
br.ranking(min_temporadas=20).head(10)   # tabela all-time (aproveitamento normalizado)

In [ ]:
br.lideres().tail(5)              # o 1º colocado dos pontos corridos de cada ano

In [ ]:
br.plot_lideres(top=10)

<a id="12"></a>
## 12. Cores oficiais dos clubes

Todo gráfico **por time** aceita `cores_times=True` (Palmeiras → verde,
Flamengo → vermelho...). O recurso é *opt-in*: cores de clube não são seguras
para daltonismo — a paleta padrão da lib é validada. Clubes fora do mapa (ou
dois clubes de mesma cor no mesmo gráfico) caem na paleta categórica.

In [ ]:
from dashgusbr import cor_time, cores_para_times

print(cor_time("Palmeiras"), cor_time("GRÊMIO"), cor_time("atletico-mg"))  # tolerante
print(cor_time("Time Inexistente", padrao="#999999"))                      # nunca quebra
cores_para_times(["Corinthians", "Botafogo", "Santos"])   # 3 alvinegros: só o 1º fica preto

<a id="13"></a>
## 13. Personalização: todas as combinações

Todo `plot_*` aceita:

| Parâmetro | Efeito |
|---|---|
| `titulo=` | sobrescreve o título padrão |
| `**layout_kwargs` | vão para `fig.update_layout` (`width`, `height`, `font`, `template`...) |
| `cores_times=True` | cores oficiais (gráficos por time) |
| `mostrar_valores=False` | esconde rótulos de valor (gráficos de barra) |
| `mostrar_legenda=False` | esconde a legenda (gráficos com legenda) |

E nas funções `viz.*`, `cores=` troca a paleta. A figura retornada é um
`plotly.graph_objects.Figure` normal — qualquer pós-processamento vale.

In [ ]:
br.plot_tabela(2023, titulo="Meu título", width=900, height=600)

In [ ]:
br.plot_tabela(2023, template="dashgusbr_escuro")     # tema escuro da lib

In [ ]:
br.plot_casa_fora("Grêmio", mostrar_valores=False, mostrar_legenda=False)

In [ ]:
fig = br.plot_gols_por_temporada(template="plotly_dark")   # qualquer tema Plotly
fig.update_traces(line_color="#ff5722")                    # pós-processamento livre
fig.add_annotation(text="Fonte: OBT Infra-Brasileirao",
                   xref="paper", yref="paper", x=1, y=-0.14, showarrow=False)
fig

In [ ]:
from dashgusbr import viz

viz.classificacao(br.tabela(2023), cores="#7a1f2b", titulo="Paleta própria via viz.*")

<a id="14"></a>
## 14. Exportação (HTML e imagem)

- `salvar_html(fig, caminho)` — página interativa (dica: `include_plotlyjs="cdn"`
  gera um arquivo ~20x menor);
- `salvar_imagem(fig, caminho)` — PNG/SVG/PDF, requer `pip install dashgusbr[imagem]`.

In [ ]:
from dashgusbr import salvar_html

destino = salvar_html(br.plot_tabela(2023), "tabela_2023.html", include_plotlyjs="cdn")
print(f"salvo em {destino} ({destino.stat().st_size / 1024:.0f} KB)")

In [ ]:
# Requer o extra [imagem]; sem ele, o erro explica como instalar:
# from dashgusbr import salvar_imagem
# salvar_imagem(fig, "grafico.png", escala=2)   # 2 = retina; aceita width=/height=

<a id="15"></a>
## 15. Qualidade dos dados

`br.validar()` roda checagens de consistência sobre a base carregada — útil
antes de confiar nas análises, ou para inspecionar um CSV próprio.

In [ ]:
br.validar()    # 0 problemas = OBT saudável

In [ ]:
# As checagens funcionam em qualquer DataFrame no schema canônico:
from dashgusbr import schema

sujo = br.df.head(50).copy()
sujo.loc[sujo.index[0], "resultado_mandante"] = "D"   # incoerência proposital
schema.relatorio_consistencia(sujo)

<a id="16"></a>
## 16. Uso avançado: as camadas puras

`Brasileirao` é uma fachada. Por baixo há três camadas de **funções puras**
(DataFrame → DataFrame → Figure) que você pode importar direto — por exemplo,
para montar seu próprio dashboard em Streamlit:

| Camada | Papel |
|---|---|
| `dashgusbr.data` | carga, fallback, caches (`carregar_dados`, `carregar_geojson_estados`, `limpar_cache`) |
| `dashgusbr.analytics` | agregações Pandas (`classificacao`, `confronto`, `fator_viagem`... tudo que a fachada usa) |
| `dashgusbr.viz` | figuras Plotly (recebem a saída de `analytics`) |

In [ ]:
from dashgusbr import analytics, data, viz

df  = data.carregar_dados()                     # a OBT completa, schema canônico
tab = analytics.classificacao(df, ano=2023)     # DataFrame -> DataFrame
viz.classificacao(tab, titulo="Montado com as camadas puras")

In [ ]:
# Qualquer análise da fachada existe como função pura:
analytics.media_gols_por_decada(df)

In [ ]:
data.limpar_cache()             # limpa o cache em memória
# data.limpar_cache(disco=True) # ...e também os arquivos em ~/.dashgusbr/cache

<a id="17"></a>
## 17. CLI: a lib no terminal

Sem escrever Python:

```bash
python -m dashgusbr tabela 2023                  # classificação no terminal
python -m dashgusbr tabela 2023 --html t.html    # + gráfico salvo em HTML
python -m dashgusbr goleadas 10
python -m dashgusbr ranking --min-temporadas 20 --top 10
python -m dashgusbr resumo Cruzeiro
python -m dashgusbr times 2023
python -m dashgusbr anos
python -m dashgusbr validar                      # sai com código 1 se houver problema
```

`--fonte caminho/obt.csv` aponta qualquer comando para um CSV local (offline).

In [ ]:
!python -m dashgusbr anos

---

### Para ir além

- **README** — referência rápida e schema completo da OBT;
- **`examples/demo.py`** — gera a galeria HTML com todos os gráficos;
- **ROADMAP.md** — o que vem por aí (Elo histórico, app Streamlit, docs site).